### Multi‑Class Covertype Classification With SHAP & LIME Explainability

---

In [1]:
!pip install ucimlrepo shap lime

In [2]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import shap
from lime.lime_tabular import LimeTabularExplainer

In [3]:
covertype = fetch_ucirepo(id=31)
X = pd.DataFrame(covertype.data.features)
y = covertype.data.targets
print(X.shape, y.value_counts().head())

(581012, 54) Cover_Type
2             283301
1             211840
3              35754
7              20510
6              17367
Name: count, dtype: int64


In [4]:

corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
X = X.drop(columns=to_drop)
print('Dropped due to correlation:', to_drop)

Dropped due to correlation: []


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
model = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


[[40138  2304     2     0     8     4   101]
 [ 1220 55032    94     2    80    58    14]
 [    3    94  6878    19     6   121     0]
 [    0     0    65   448     0    13     0]
 [   33   387    18     0  1548     9     0]
 [    1   102   219    19     3  3145     0]
 [  170    27     0     0     0     0  3818]]
              precision    recall  f1-score   support

           1       0.97      0.94      0.95     42557
           2       0.95      0.97      0.96     56500
           3       0.95      0.97      0.96      7121
           4       0.92      0.85      0.88       526
           5       0.94      0.78      0.85      1995
           6       0.94      0.90      0.92      3489
           7       0.97      0.95      0.96      4015

    accuracy                           0.96    116203
   macro avg       0.95      0.91      0.93    116203
weighted avg       0.96      0.96      0.96    116203



In [ ]:

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test[:50])
shap.summary_plot(shap_values, X_test[:50], feature_names=X.columns)

In [ ]:
lime_exp = LimeTabularExplainer(X_train[:200], feature_names=X.columns, class_names=np.unique(y).astype(str), discretize_continuous=True)
exp = lime_exp.explain_instance(X_test[0], model.predict_proba)
exp.show_in_notebook()